In [2]:
using MLJ
using MLJFlux                  # interfaz para modelos Flux
using MLJScikitLearnInterface  # interfaz para SVM (scikit-learn wrapper)
using NearestNeighborModels    # interfaz para KNN

In [3]:

include("P2.jl")

crossvalidation (generic function with 4 methods)

# 1. Preparación de los datos

## 1.1. Carga y unificación de datos

In [4]:
unifyDataset("./DatosPractica", "./dataset.csv") # Se leen todos los datos del árbol de carpetas y se guardan en un único csv

df = CSV.read("./dataset.csv", DataFrame) # Lectura de ese mismo csv

subjects, data, targets = separateDataframe(df)

println("Numero de variables: $(length(data[1,:]))")
println("Numero de instancias: $(length(data[:,1]))")
println("Número de sujetos: $(length(unique(subjects)))")
println("Número de clases de salida: $(length(unique(targets)))")


Numero de variables: 561
Numero de instancias: 10299
Número de sujetos: 30
Número de clases de salida: 6


## 1.2. Análisis de valores ausentes

In [5]:
missings = sum(sum(ismissing.(x) for x in eachcol(data)))
missingPercentage = (missings*100)/(length(data[1,:])*length(data[:,1]))
println("Porcentaje de valores faltantes: $(round(missingPercentage, sigdigits = 5))%")

Porcentaje de valores faltantes: 1.002%


## 1.3. Tratamiento de datos

In [6]:
# Eliminar missings del dataset
replaceWithMean!(data)

missings = sum(sum(ismissing.(x) for x in eachcol(data)))
missingPercentage = (missings*100)/(length(data[1,:])*length(data[:,1]))
println("Porcentaje de valores faltantes: $(round(missingPercentage, sigdigits = 5))%")

# Convertir el dataset a Array para trabajar más fácil con él
data = Array{Float32}(data)

Porcentaje de valores faltantes: 0.0%


10299×561 Matrix{Float32}:
 -0.361205   -0.268121     0.176896   …   0.508733  -0.496132  -0.506197
 -0.277066   -0.684097     0.346658       0.668284  -0.336641  -0.672685
 -0.239103   -0.0969044    0.148035       0.378792  -0.488443  -0.487057
 -0.166426   -0.119353     0.133035       0.272083  -0.534122  -0.397428
 -0.0832871   0.0102977   -0.516084       0.364277  -0.804865   0.144034
 -0.0662389   0.079229    -1.0        …   0.269545  -0.754285  -0.0974772
 -0.0417013   0.175102     0.0255518     -0.552939  -0.053539  -0.260424
  0.0139037   0.153296     0.0162429     -0.418368  -0.142549  -0.305884
  0.0190162  -0.00703736  -0.0283334      0.205759  -0.536104  -0.360736
  0.130461   -0.0549969   -0.131786      -0.766224   0.258457   0.03835
  ⋮                                   ⋱                         ⋮
  0.390494   -0.0186085   -0.0877924  …  -0.856499   0.197149   0.0107762
  0.396915   -0.0221819   -0.234221      -0.89164    0.149776  -0.0393991
  0.397628   -0.01348     -0.

In [7]:
targets = oneHotEncoding(targets)

10299×6 BitMatrix:
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 0  1  0  0  0  0
 0  1  0  0  0  0
 1  0  0  0  0  0
 0  0  1  0  0  0
 ⋮              ⋮
 0  0  0  0  1  0
 0  0  1  0  0  0
 0  0  0  0  1  0
 0  0  1  0  0  0
 0  0  1  0  0  0
 0  0  0  1  0  0
 0  0  1  0  0  0
 0  0  1  0  0  0
 0  0  1  0  0  0

## 1.4. Partición Holdout

In [8]:
using Random

Random.seed!(104)

trainSubjectNumbers, testSubjectNumbers = holdOut(length(unique(subjects)), 0.1)

println("Test subjects: $testSubjectNumbers")

testIndices = findall(x-> x in testSubjectNumbers, subjects)

testData = data[testIndices,:]
testTargets = targets[testIndices,:]
testSubjects = subjects[testIndices,:]
println(unique(testSubjects))

println("Train subjects: $(sort(trainSubjectNumbers))")

trainIndices = findall(x-> x in trainSubjectNumbers, subjects)

trainData = data[trainIndices,:]
trainTargets = targets[trainIndices,:]
trainSubjects = subjects[trainIndices,:]
println(unique(trainSubjects))

Test subjects: [8, 24, 26]
[8, 24, 26]
Train subjects: [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 27, 28, 29, 30]
[1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 27, 28, 29, 30]


## 1.5. Validación cruzada individual-wise

In [9]:
using Random

Random.seed!(104)

# No se si esta bien esto, no es subject wise
crossValidationSubjects = crossvalidation(length(trainSubjectNumbers),5)

27-element Vector{Int64}:
 3
 4
 1
 5
 4
 2
 3
 5
 3
 1
 ⋮
 2
 4
 4
 1
 3
 1
 2
 3
 1

## 1.6. Normalización

In [ ]:
# por probar puedes borrarlo y dejarlo como estaba antes
norm_param = calculateMinMaxNormalizationParameters(trainData)
normalizeMinMax!(trainData,norm_param)
# normalizeMinMax!(valData)
normalizeMinMax!(testData,norm_param)


1054×561 Matrix{Float32}:
 0.0       0.122728  0.891792  0.405077   …  0.781685   0.515889  0.0793253
 0.43245   0.0       0.809045  0.384099      0.895228   0.455073  0.103607
 0.490256  0.957163  0.590124  0.139715      0.297405   0.553468  0.498876
 0.542477  0.244111  0.472018  0.55205       0.236403   0.94739   0.972
 0.590923  0.150599  0.486644  0.593788      0.229958   0.946672  0.964958
 0.596152  0.639879  0.789565  0.0744166  …  0.215246   0.668541  0.521992
 0.598744  0.366234  0.439419  0.681092      0.141849   0.986983  0.74459
 0.599804  0.435853  0.352551  0.619687      0.231853   0.939273  0.971421
 0.600455  0.235387  0.569769  0.571311      0.121493   0.963666  0.735579
 0.606604  0.33396   0.582889  0.549003      0.13872    0.983093  0.754522
 ⋮                                        ⋱                       ⋮
 0.894071  0.348798  0.528251  0.5283     …  0.0539696  0.855342  0.679943
 0.9043    0.336722  0.547093  0.532534      0.0511663  0.85507   0.683561
 0.905066

In [11]:
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux
model = NeuralNetworkClassifier([50])
mach = machine(model, trainData, trainTargets)
fit!(mach)

┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\rodga\.julia\packages\MLJModels\ziReN\src\loading.jl:159


import MLJFlux ✔


MethodError: MethodError: no method matching MLJFlux.NeuralNetworkClassifier(::Vector{Int64})
The type `MLJFlux.NeuralNetworkClassifier` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  MLJFlux.NeuralNetworkClassifier(::B, !Matched::F, !Matched::O, !Matched::L, !Matched::Int64, !Matched::Int64, !Matched::Float64, !Matched::Float64, !Matched::Union{Int64, AbstractRNG}, !Matched::Bool, !Matched::ComputationalResources.AbstractResource, !Matched::Dict{Symbol, Real}) where {B, F, O, L}
   @ MLJFlux C:\Users\rodga\.julia\packages\MLJFlux\5eWpt\src\types.jl:17
  MLJFlux.NeuralNetworkClassifier(; builder, finaliser, optimiser, loss, epochs, batch_size, lambda, alpha, rng, optimiser_changes_trigger_retraining, acceleration, embedding_dims)
   @ MLJFlux C:\Users\rodga\.julia\packages\MLJFlux\5eWpt\src\types.jl:31


In [26]:
SVMClassifier = @load SVMClassifier pkg=MLJScikitLearnInterface verbosity=2
model = SVMClassifier(C=1)
mach = machine(model, trainData, trainTargets)
fit!(mach)

import MLJScikitLearnInterface ✔


┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\rodga\.julia\packages\MLJModels\ziReN\src\loading.jl:159
┌ Warning: The number and/or types of data arguments do not match what the specified model
│ supports. Suppress this type check by specifying `scitype_check_level=0`.
│ 
│ Run `@doc MLJScikitLearnInterface.SVMClassifier` to learn more about your model's requirements.
│ 
│ Commonly, but non exclusively, supervised models are constructed using the syntax
│ `machine(model, X, y)` or `machine(model, X, y, w)` while most other models are
│ constructed with `machine(model, X)`.  Here `X` are features, `y` a target, and `w`
│ sample or class weights.
│ 
│ In general, data in `machine(model, data...)` is expected to satisfy
│ 
│     scitype(data) <: MLJ.fit_data_scitype(model)
│ 
│ In the present case:
│ 
│ scitype(data) = Tuple{AbstractMatrix{Continuous}, AbstractMatrix{Count}}
│ 
│ fit_data_scitype(model) = Tuple{Table{<:AbstractVector{<:Continuous}}, AbstractVector{

DomainError: DomainError with true:
Can only convert categorical elements to integers. 

In [32]:
# Entrenar el modelo KNN con el formato correcto
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels
model = KNNClassifier(K=100)
mach = machine(model, trainData, trainTargets)
fit!(mach)

# Hacer predicciones
predictions = predict(mach, testData)

accuracy_score = accuracy(predictions, testTargets)
println("Accuracy: $accuracy_score")

import NearestNeighborModels ✔


┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\rodga\.julia\packages\MLJModels\ziReN\src\loading.jl:159
┌ Warning: The number and/or types of data arguments do not match what the specified model
│ supports. Suppress this type check by specifying `scitype_check_level=0`.
│ 
│ Run `@doc NearestNeighborModels.KNNClassifier` to learn more about your model's requirements.
│ 
│ Commonly, but non exclusively, supervised models are constructed using the syntax
│ `machine(model, X, y)` or `machine(model, X, y, w)` while most other models are
│ constructed with `machine(model, X)`.  Here `X` are features, `y` a target, and `w`
│ sample or class weights.
│ 
│ In general, data in `machine(model, data...)` is expected to satisfy
│ 
│     scitype(data) <: MLJ.fit_data_scitype(model)
│ 
│ In the present case:
│ 
│ scitype(data) = Tuple{AbstractMatrix{Continuous}, AbstractMatrix{Count}}
│ 
│ fit_data_scitype(model) = Union{Tuple{Table{<:AbstractVector{<:Continuous}}, AbstractVec

MethodError: MethodError: no method matching classes(::Bool)
The function `classes` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  classes(!Matched::Union{UnivariateFinite{S, V, R, P}, CategoricalDistributions.UnivariateFiniteArray{S, V, R, P}} where {S, V, R, P})
   @ CategoricalDistributions C:\Users\rodga\.julia\packages\CategoricalDistributions\g0UnN\src\methods.jl:17
  classes(!Matched::SubArray{<:Any, <:Any, <:CategoricalArrays.CategoricalArray})
   @ CategoricalDistributions C:\Users\rodga\.julia\packages\CategoricalDistributions\g0UnN\src\utilities.jl:61
  classes(!Matched::CategoricalArrays.CategoricalArray)
   @ CategoricalDistributions C:\Users\rodga\.julia\packages\CategoricalDistributions\g0UnN\src\utilities.jl:60
  ...
